In [5]:
# Modeling Pipeline for AgriShield AI
# Load processed feature data, train classification and regression models,
# evaluate performance, add SHAP explainability, and export model artifacts.

import os
import sys
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Optional libraries
try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
except ImportError:
    CatBoostClassifier = None
    CatBoostRegressor = None

try:
    import shap
except ImportError:
    shap = None

# Paths
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(ROOT_DIR, 'Data', 'processed')
MODEL_DIR = os.path.join(ROOT_DIR, 'modelML', 'artifacts')

os.makedirs(MODEL_DIR, exist_ok=True)

FEATURE_FILE = os.path.join(DATA_DIR, 'feature_dataset.csv')
FSRI_FILE = os.path.join(DATA_DIR, 'fsri_dataset.csv')

print(f'Feature dataset: {FEATURE_FILE}')
print(f'FSRI dataset: {FSRI_FILE}')
print(f'Model artifacts directory: {MODEL_DIR}')

# Load data
feature_df = pd.read_csv(FEATURE_FILE)
fsri_df = pd.read_csv(FSRI_FILE)

# Merge on Country and Year to attach FSRI targets
model_df = feature_df.merge(
    fsri_df[['Country', 'Year', 'FSRI', 'FSRI_Category']],
    on=['Country', 'Year'],
    how='inner'
)

print(f'Loaded model dataset: {model_df.shape[0]} rows, {model_df.shape[1]} columns')

# Define features and targets
feature_columns = [
    col for col in feature_df.columns
    if col not in ['Country', 'Year']
]

X = model_df[feature_columns].copy()
y_class = model_df['FSRI_Category'].copy()
y_reg = model_df['FSRI'].copy()

# Encode labels
label_encoder = LabelEncoder()
y_class_encoded = label_encoder.fit_transform(y_class)

# Save feature list and label encoder later
feature_list = feature_columns

# Evaluate class distribution and decide stratification strategy
class_counts = np.bincount(y_class_encoded)
class_distribution = dict(zip(label_encoder.classes_, class_counts))
print('Class distribution:', class_distribution)

stratify_target = None
if np.min(class_counts) >= 2:
    stratify_target = y_class_encoded
else:
    print('WARNING: Some classes have fewer than 2 samples, disabling stratified split.')

# Train/test split with optional stratification
X_train, X_test, y_train_cat, y_test_cat, y_train_reg, y_test_reg = train_test_split(
    X, y_class_encoded, y_reg, test_size=0.2, stratify=stratify_target, random_state=42
)

print('Training samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])

# Preprocessing pipeline
numeric_features = feature_columns
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ],
    remainder='passthrough'
)

# Utility functions

def build_classifier_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])


def build_regressor_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])


def evaluate_classification(name, y_true, y_pred, y_proba=None):
    results = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0)
    }
    if y_proba is not None:
        if y_proba.ndim == 2 and y_proba.shape[1] == len(label_encoder.classes_):
            try:
                results['roc_auc_ovo'] = roc_auc_score(y_true, y_proba, multi_class='ovo')
            except ValueError as e:
                print(f'WARNING: ROC AUC skipped for {name}: {e}')
        else:
            print(f'WARNING: ROC AUC skipped for {name}: probability output shape {y_proba.shape}')
    print(f'\n{name} Classification Results:')
    for metric, value in results.items():
        print(f'  {metric}: {value:.4f}')
    print('  Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    return results


def evaluate_regression(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    results = {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mse),
        'r2': r2_score(y_true, y_pred)
    }
    print(f'\n{name} Regression Results:')
    for metric, value in results.items():
        print(f'  {metric}: {value:.4f}')
    return results

# Baseline models
classification_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
}

regression_models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=200, random_state=42)
}

if xgb is not None:
    classification_models['XGBoost'] = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    regression_models['XGBoostRegressor'] = xgb.XGBRegressor(random_state=42)

if lgb is not None:
    classification_models['LightGBM'] = lgb.LGBMClassifier(random_state=42)
    regression_models['LightGBMRegressor'] = lgb.LGBMRegressor(random_state=42)

if CatBoostClassifier is not None:
    classification_models['CatBoost'] = CatBoostClassifier(verbose=0, random_state=42)
if CatBoostRegressor is not None:
    regression_models['CatBoostRegressor'] = CatBoostRegressor(verbose=0, random_state=42)

# Train baseline classifiers
classification_results = {}
for name, model in classification_models.items():
    print(f'\nTraining classifier: {name}')
    pipeline = build_classifier_pipeline(model)
    pipeline.fit(X_train, y_train_cat)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test) if hasattr(pipeline, 'predict_proba') else None
    classification_results[name] = evaluate_classification(name, y_test_cat, y_pred, y_proba)
    joblib.dump(pipeline, os.path.join(MODEL_DIR, f'{name}.pkl'))

# Train baseline regressors
regression_results = {}
for name, model in regression_models.items():
    print(f'\nTraining regressor: {name}')
    pipeline = build_regressor_pipeline(model)
    pipeline.fit(X_train, y_train_reg)
    y_pred = pipeline.predict(X_test)
    regression_results[name] = evaluate_regression(name, y_test_reg, y_pred)
    joblib.dump(pipeline, os.path.join(MODEL_DIR, f'{name}.pkl'))

# Save artifacts
joblib.dump(preprocessor, os.path.join(MODEL_DIR, 'preprocessor.pkl'))
joblib.dump(label_encoder, os.path.join(MODEL_DIR, 'label_encoder.pkl'))
joblib.dump(classification_results, os.path.join(MODEL_DIR, 'classification_results.pkl'))
joblib.dump(regression_results, os.path.join(MODEL_DIR, 'regression_results.pkl'))
with open(os.path.join(MODEL_DIR, 'feature_list.json'), 'w', encoding='utf-8') as f:
    json.dump(feature_list, f, indent=2)

print(f'Artifacts saved to {MODEL_DIR}')


Feature dataset: c:\Users\akilm\Documents\UTU\Data\processed\feature_dataset.csv
FSRI dataset: c:\Users\akilm\Documents\UTU\Data\processed\fsri_dataset.csv
Model artifacts directory: c:\Users\akilm\Documents\UTU\modelML\artifacts
Loaded model dataset: 2816 rows, 22 columns
Class distribution: {'Critical': np.int64(1), 'High Risk': np.int64(224), 'Medium': np.int64(508), 'Safe': np.int64(1463), 'Very Safe': np.int64(620)}
Training samples: 2252
Test samples: 564

Training classifier: LogisticRegression

LogisticRegression Classification Results:
  accuracy: 0.9486
  precision_macro: 0.9252
  recall_macro: 0.9635
  f1_macro: 0.9420
  Confusion Matrix:
[[ 47   0   0   0]
 [  7 104   0   1]
 [  0   6 261  15]
 [  0   0   0 123]]

Training classifier: RandomForest

RandomForest Classification Results:
  accuracy: 0.9574
  precision_macro: 0.9654
  recall_macro: 0.9494
  f1_macro: 0.9570
  Confusion Matrix:
[[ 45   2   0   0]
 [  1 107   4   0]
 [  0   1 277   4]
 [  0   0  12 111]]

Trainin